In [7]:
import os, importlib#, netCDF4
import numpy as np
import scipy
import matplotlib.pyplot as plt

#import Model
#import PyPreprocess

In [8]:
src_dir = '/mnt/c/Users/user/Desktop/slc_cache/251114rGreenland/'
s1_fn = src_dir + "ALPSRP060811420-L1.1.zip"
s2_fn = src_dir + "ALPSRP067521420-L1.1.zip"
stk_fn = '/mnt/c/Users/user/Desktop/slc_cache/tmp/ALOS-P1_1__A-ORBIT__ALPSRPstack'

In [9]:
from esa_snappy import snapista

graph = snapista.Graph()

# Read
graph.add_node(operator=snapista.Operator('Read', file=s1_fn), node_id = 'read_s1')
graph.add_node(operator=snapista.Operator('Read', file=s2_fn), node_id = 'read_s2')

bnd_used = ['i_HH','q_HH','i_HV','q_HV','i_VH','q_VH','i_VV','q_VV']
dem_used = 'Copernicus 30m Global DEM'

# Deskew
oper1 = snapista.Operator('ALOS-Deskewing', demName=dem_used)
graph.add_node(operator = oper1, node_id = 'deskew_s1', source = 'read_s1')
graph.add_node(operator = oper1, node_id = 'deskew_s2', source = 'read_s2')

# Calibration
oper1 = snapista.Operator('Calibration', outputImageInComplex='true', outputSigmaBand='false')
graph.add_node(operator = oper1, node_id = 'calib_s1', source = 'deskew_s1')
graph.add_node(operator = oper1, node_id = 'calib_s2', source = 'deskew_s2')
#oper2 = snapista.Operator('Faraday-Rotation-Correction') - Faraday rotation bugged: i_VV is always zeroed
#graph.add_node(operator = oper2, node_id = 'calibb_s1', source = 'calib_s1')
#graph.add_node(operator = oper2, node_id = 'calibb_s2', source = 'calib_s2')

# Coregistration
oper1 = snapista.Operator('CreateStack', resamplingType='NEAREST_NEIGHBOUR', initialOffsetMethod='Product Geolocation')
oper2 = snapista.Operator('Cross-Correlation', numGCPtoGenerate='1000', coherenceWindowSize='5', coherenceThreshold='0.5', useSlidingWindow='true')
oper3 = snapista.Operator('Warp', interpolationMethod='Bilinear interpolation', demName=dem_used)
graph.add_node(operator = oper1, node_id = 'stk_op1', source = ['calib_s1','calib_s2'])
graph.add_node(operator = oper2, node_id = 'stk_op2', source = 'stk_op1')
graph.add_node(operator = oper3, node_id = 'stk_op3', source = 'stk_op2')

#select relevant bands & write to file
oper1 = snapista.Operator('BandSelect', bandNamePattern='^._([HVRL]{2})_.*') # follows pattern i_PP_...
oper2 = snapista.Operator('Write', file=stk_fn, formatName='NETCDF4-BEAM')
graph.add_node(operator = oper1, node_id = 'stk_op4', source = 'stk_op3')
graph.add_node(operator = oper2, node_id = 'stk_write', source = 'stk_op4')

In [10]:
graph.run()

Processing the graph, this may take a while. Please wait...
Executing processing graph

100% done.

100% done.

100% done.

100% done.
....10%....21%...UTC parse error:2007-3-16 22:44:1260.0:java.lang.NumberFormatException: multiple points
UTC parse error:2007-3-16 22:44:360.0:java.lang.NumberFormatException: multiple points
.31%....42%....52%....62%....72%....82%.... done.
INFO: org.esa.snap.core.gpf.operators.tooladapter.ToolAdapterIO: Initializing external tool adapters
INFO: org.esa.snap.core.util.EngineVersionCheckActivator: Please check regularly for new updates for the best SNAP experience.
42528 [main] INFO serverStartup - Nc4Iosp: NetCDF-4 C library loaded (jna_path='/home/adminix/.snap/auxdata/netcdf_natives/12.0.0/amd64', libname='netcdf').
42531 [main] INFO serverStartup - NetcdfLoader: set log level: old=0 new=0
42531 [main] INFO serverStartup - Nc4Iosp: set log level: old=0 new=0
-- org.jblas INFO Deleting /tmp/jblas247337758345644690/libquadmath-0.so
-- org.jblas INFO De

0